In [1]:
import pandas as pd
import json
from itertools import combinations
import numpy as np
import torch
import re
from typing import Iterator
from torch.utils.data import DataLoader
from datasets import Dataset
from sentence_transformers import models, SentenceTransformer, InputExample
from sentence_transformers.evaluation import TripletEvaluator, BinaryClassificationEvaluator, InformationRetrievalEvaluator, SequentialEvaluator
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss, TripletLoss, ContrastiveLoss
from sentence_transformers.sampler import GroupByLabelBatchSampler
from torch.utils.data import BatchSampler
from torch import distributed as dist
from peft import LoraConfig, TaskType, get_peft_model
import sys
from datetime import datetime
from transformers import BitsAndBytesConfig

/u/modelfactory/.conda/envs/lab/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
try:
    dist.init_process_group(backend="nccl", init_method="env://")
    my_rank = dist.get_rank()
    device = my_rank
except:
    device = 'cuda'
print(device)

cuda


In [3]:
model_id = 'google-bert/bert-base-uncased'
# model_id = 'McGill-NLP/LLM2Vec-Meta-llama-31-8B-Instruct-mntp-supervised'
# model_id = 'nvidia/NV-Embed-v1'
# model_id = 'Alibaba-NLP/gte-Qwen2-7B-instruct'
# model_id = 'ibm-granite/granite-3.1-8b-instruct'
batch_size = 32
pooling_mode = 'mean'
# p_desc None means the original experiment (not the ablation folder)
p_desc = 0.0
positive_samples_ratio = 0.25
seed_no = 42
if sys.argv[1] != '-f' and sys.argv[4] != 'None':
    model_id = sys.argv[1]
    batch_size = int(sys.argv[2])
    pooling_mode = sys.argv[3]
    p_desc = float(sys.argv[4])
    positive_samples_ratio = float(sys.argv[5])
    seed_no = int(sys.argv[6])
# model_id = 'nvidia/NV-Embed-v2'
# model_id = 'intfloat/e5-mistral-7b-instruct'
# model_id = 'BAAI/bge-large-en-v1.5'
# model_id = 'models/intfloat_e5-mistral-7b-instruct_fmsr_ft/checkpoint-1895'

In [4]:
np.random.seed(seed_no)

In [5]:
print(f'batch_size:{batch_size}, seed:{seed_no}, p_desc:{p_desc}, positive_ratio:{positive_samples_ratio}')

batch_size:32, seed:42, p_desc:0.0, positive_ratio:0.25


In [6]:
lora_models = [
    'Alibaba-NLP/gte-Qwen2-7B-instruct', 
    'intfloat/e5-mistral-7b-instruct', 
    'McGill-NLP/LLM2Vec-Meta-llama-31-8B-Instruct-mntp-supervised',
    'ibm-granite/granite-3.1-8b-instruct'
    # 'nvidia/NV-Embed-v1'
    # 'nvidia/NV-Embed-v2',
    # '/dccstor/dcpfactory/.cache/huggingface/hub/models--nvidia--NV-Embed-v2/snapshots/5130cf1daf847c1bacee854a6ef1ca939e747fb2'
]

In [7]:
print(f'model_id: {model_id}, batch_size: {batch_size}, pooling_mode:{pooling_mode}, p_desc:{p_desc}')

model_id: google-bert/bert-base-uncased, batch_size: 32, pooling_mode:mean, p_desc:0.0


In [8]:
kwargs = {'torch_dtype': torch.bfloat16} if model_id in lora_models or 'NV' in model_id else {}
kwargs['trust_remote_code'] = True
if pooling_mode == 'mean':
    print('loading with mean pooling mode')
    model = SentenceTransformer(model_id, device=device, trust_remote_code=True, model_kwargs=kwargs)
else:
    print('loading with last pooling mode')
    transformer = models.Transformer(model_id, model_args=kwargs)
    pooling = models.Pooling(transformer.get_word_embedding_dimension(), pooling_mode="lasttoken")
    normalize = models.Normalize()
    model = SentenceTransformer(modules=[transformer, pooling, normalize], device=device, trust_remote_code=True, model_kwargs=kwargs)

loading with mean pooling mode


No sentence-transformers model found with name google-bert/bert-base-uncased. Creating a new one with mean pooling.


In [9]:
# # lora for big models like e5-mistral-7b
if model_id in lora_models:
    print('doing lora')
    peft_config = LoraConfig(
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type=TaskType.FEATURE_EXTRACTION,
        inference_mode=False,
        r=8,
        lora_alpha=32,
        lora_dropout=0.1,
    )
    
    model._modules["0"].auto_model = get_peft_model(
        model._modules["0"].auto_model, peft_config
    )

## Load train/val/test dataset

In [6]:
base_data_path = 'data/industrial/processed'
if p_desc is not None:
    base_data_path = f'data/industrial/processed/ablation/p_desc_{p_desc}_seed_{seed_no}'

In [7]:
train_dataset = Dataset.from_json(f'{base_data_path}/train.json')
val_dataset = Dataset.from_json(f'{base_data_path}/val.json')
test_dataset = Dataset.from_json(f'{base_data_path}/test.json')
rename_cols = {'sentence1': 'anchor', 'sentence2': 'positive'}
train_dataset_multineg = train_dataset.filter(lambda x: x['label']==1).remove_columns(['label']).rename_columns(rename_cols)
val_dataset_multineg = val_dataset.filter(lambda x: x['label']==1).remove_columns(['label']).rename_columns(rename_cols)
# test_dataset = test_dataset.filter(lambda x: x['label']==1).remove_columns(['label']).rename_columns(rename_cols)

In [8]:
len(train_dataset), len(val_dataset), len(test_dataset)

(127464, 49460, 42982)

In [9]:
with open(f'{base_data_path}/all_tasks_dataset.json', 'r') as f:
    data = json.load(f)
id_to_corpus = {val: key for key, val in data['corpus_to_id'].items()}
id_to_query = {val: key for key, val in data['query_to_id'].items()}
query_to_doc = {int(key): val for key, val in data['query_to_doc'].items()}

In [10]:
unique_tasks = list(data['dataset_to_queries'].keys())
task_dataset = {'val': {}, 'test': {}}
for task in unique_tasks:
    task_dataset['val'][task] = val_dataset.filter(lambda x: x['task'] == task)
for task in unique_tasks:
    task_dataset['test'][task] = test_dataset.filter(lambda x: x['task'] == task)

## Prepare evaluators for val and test

In [11]:
class ScoreAggregator:
    def __init__(self, datasets):
        lengths = [len(dataset) for dataset in datasets]
        total_size = sum(lengths)
        self.proportions = [length / total_size for length in lengths]
    def __call__(self, scores):
        # weighted average
        return sum([score * proportion for score, proportion in zip(scores, self.proportions)])

In [12]:
agg = {'val': ScoreAggregator(val_dataset), 'test': ScoreAggregator(test_dataset)}

In [13]:
evaluators = {'val': [], 'test': []}
for split in ['val', 'test']:
    for task in task_dataset[split]:
        task_queries = {data['query_to_id'][item['sentence1']]: item['sentence1'] for item in task_dataset[split][task]}
        evaluator = InformationRetrievalEvaluator(
            queries=task_queries,
            corpus=id_to_corpus,
            relevant_docs=query_to_doc,
            name=f"{task}_{split}"
        )
        evaluators[split].append(evaluator)
    evaluators[split] = SequentialEvaluator(evaluators[split], main_score_function=agg[split])

In [18]:
scores_before = evaluators['test'](model)

In [52]:
class BalancedSampler(GroupByLabelBatchSampler):
    def __iter__(self) -> Iterator[list[int]]:
        if self.generator and self.seed:
            self.generator.manual_seed(self.seed + self.epoch)

        partial_batch = []
        unique_labels = list(self.groups.keys())
        max_group_size = max(len(self.groups[label]) for label in self.groups)
        for i in np.random.permutation(max_group_size):
            for label_idx in self.groups:
                if i >= len(self.groups[label_idx]):
                    self.groups[label_idx].extend(np.random.choice(self.groups[label_idx], i - len(self.groups[label_idx]) + 1))
                partial_batch.append(self.groups[label_idx][i])
                while len(partial_batch) >= self.batch_size:
                    yield partial_batch[: self.batch_size]
                    partial_batch = partial_batch[self.batch_size :]

        if not self.drop_last and partial_batch:
            yield partial_batch

In [67]:
class BalancedSamplerAblation(GroupByLabelBatchSampler):
    def __iter__(self) -> Iterator[list[int]]:
        if self.generator and self.seed:
            self.generator.manual_seed(self.seed + self.epoch)

        partial_batch = []
        unique_labels = list(self.groups.keys())
        negative_group_size = len(self.groups[0])
        positive_group_size = len(self.groups[1])
        positive_samples_per_batch = int(positive_samples_ratio * self.batch_size)
        positive_permutations = np.random.permutation(positive_group_size)
        negative_permutations = np.random.permutation(negative_group_size)
        positive_idx = 0
        negative_idx = 0
        iter_batch = 0
        while iter_batch < self.__len__():
            # add positives
            for _ in range(positive_samples_per_batch):
                # no more samples
                if positive_idx >= len(self.groups[1]):
                    # reset index and reshufle
                    positive_permutations = np.random.permutation(positive_group_size)
                    positive_idx = 0
                if positive_permutations[positive_idx] >= len(self.groups[1]):
                    self.groups[1].extend(np.random.choice(self.groups[1], positive_permutations[positive_idx] - len(self.groups[1]) + 1))
                partial_batch.append(self.groups[1][positive_permutations[positive_idx]])
                positive_idx += 1
            for _ in range(self.batch_size - positive_samples_per_batch):
                if negative_idx >= len(self.groups[0]):
                    # reset index and reshufle
                    negative_permutations = np.random.permutation(negative_group_size)
                    negative_idx = 0
                if negative_permutations[negative_idx] >= len(self.groups[0]):
                    self.groups[0].extend(np.random.choice(self.groups[0], negative_permutations[negative_idx] - len(self.groups[0]) + 1))
                partial_batch.append(self.groups[0][negative_permutations[negative_idx]])
                negative_idx += 1
            # add negatives
            yield partial_batch[: self.batch_size]
            partial_batch = partial_batch[self.batch_size:]
            iter_batch += 1
        if not self.drop_last and partial_batch:
            yield partial_batch

In [71]:
# sampler = BalancedSampler(test_dataset, 32, False, valid_label_columns=['label'])
# sampler = BalancedSamplerAblation(train_dataset, 32, False, valid_label_columns=['label'])

In [72]:
# for i, item in enumerate(sampler):
#     if i %100 == 0:
#         print(sum([train_dataset[x]['label'] for x in item]))

In [22]:
ckpt_path = f"models/{model_id.replace('/', '_')}_fmsr_ft_bs{batch_size}_pool{pooling_mode}_pdesc{p_desc}_positiveratio_{positive_samples_ratio}seed_{seed_no}"
print(ckpt_path)

models/google-bert_bert-base-uncased_fmsr_ft_bs32_poolmean_pdesc0.0_positiveratio_0.25seed_42


In [23]:
# multi_neg_loss = MultipleNegativesRankingLoss(model)
contrastive_loss = ContrastiveLoss(model)

In [24]:
class CustomTrainer(SentenceTransformerTrainer):
    def get_batch_sampler(
        self,
        *args,
        **kwargs
    ) -> BatchSampler | None:
        if p_desc is None or positive_samples_ratio==0.5:
            return BalancedSampler(*args, **kwargs)
        return BalancedSamplerAblation(*args, **kwargs)

In [30]:
args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir=ckpt_path,
    # Optional training parameters:
    do_eval=True,
    num_train_epochs=3,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    # bf16=True,  # Set to True if you have a GPU that supports BF16
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=1,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model='sequential_score',
    greater_is_better=True,
    run_name=ckpt_path.replace('/', '_'),  # Will be used in W&B if `wandb` is installed
    # batch_sampler=BatchSamplers.BALANCED_SAMPLER
)

## Then train with contrastive loss to separate within tasks

In [ ]:
trainer = CustomTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset.remove_columns(['task']),
    eval_dataset=val_dataset.remove_columns(['task']),
    evaluator=evaluators['val'],
    loss=contrastive_loss,
)
trainer.train()

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
wandb: Currently logged in as: cc4718. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss


In [30]:
# # # # test before and after finetuning
# kwargs = {'torch_dtype': torch.bfloat16}
# kwargs['trust_remote_code'] = True

# # model_id_before = 'ibm-granite/granite-3.1-8b-instruct'
# # model_id_after = 'models/ibm-granite_granite-3.1-8b-instruct_fmsr_ft/checkpoint-1000'

# # model_id_before = 'intfloat/e5-mistral-7b-instruct'
# # model_id_after = 'models/intfloat_e5-mistral-7b-instruct_fmsr_ft/checkpoint-3900'

# model_id_before = 'Alibaba-NLP/gte-Qwen2-7B-instruct'
# model_id_after = 'models/Alibaba-NLP_gte-qwen2-7B-instruct_fmsr_ft/checkpoint-3400'

# transformer = models.Transformer(model_id_before, model_args=kwargs)
# pooling = models.Pooling(transformer.get_word_embedding_dimension(), pooling_mode="lasttoken")
# normalize = models.Normalize()
# model = SentenceTransformer(modules=[transformer, pooling, normalize], device=device, trust_remote_code=True, model_kwargs=kwargs)
# # model = SentenceTransformer(model_id_before, device=device, trust_remote_code=True, model_kwargs=kwargs)
# print('loading model before')
# scores_before = evaluators['test'](model)
# print('evaluation model before')
# # model = SentenceTransformer(model_id_after, device=device, trust_remote_code=True, model_kwargs=kwargs)
# transformer = models.Transformer(model_id_after, model_args=kwargs)
# pooling = models.Pooling(transformer.get_word_embedding_dimension(), pooling_mode="lasttoken")
# normalize = models.Normalize()
# model = SentenceTransformer(modules=[transformer, pooling, normalize], device=device, trust_remote_code=True, model_kwargs=kwargs)
# print('loading model after')
# scores_after = evaluators['test'](model)
# print('evaluation model after')

Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:34<00:00,  4.89s/it]


loading model before
evaluation model before


Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:04<00:00,  1.57it/s]


loading model after
evaluation model after


In [51]:
scores_after = evaluators['test'](model)

In [31]:
scores_dict = {}
for key in scores_after:
    scores_dict[key + '_before'] = round(scores_before[key] * 100, 2)
    scores_dict[key + '_after'] = round(scores_after[key] * 100, 2)
    print(f'{key}, before: {scores_before[key]}, after:{scores_after[key]}')

eq_to_category_test_cosine_accuracy@1, before: 0.0, after:0.8888888888888888
eq_to_category_test_cosine_accuracy@3, before: 0.0, after:1.0
eq_to_category_test_cosine_accuracy@5, before: 0.0, after:1.0
eq_to_category_test_cosine_accuracy@10, before: 0.4444444444444444, after:1.0
eq_to_category_test_cosine_precision@1, before: 0.0, after:0.8888888888888888
eq_to_category_test_cosine_precision@3, before: 0.0, after:0.7777777777777778
eq_to_category_test_cosine_precision@5, before: 0.0, after:0.711111111111111
eq_to_category_test_cosine_precision@10, before: 0.05555555555555555, after:0.5333333333333333
eq_to_category_test_cosine_recall@1, before: 0.0, after:0.11851851851851852
eq_to_category_test_cosine_recall@3, before: 0.0, after:0.288888888888889
eq_to_category_test_cosine_recall@5, before: 0.0, after:0.4444444444444444
eq_to_category_test_cosine_recall@10, before: 0.037037037037037035, after:0.5925925925925926
eq_to_category_test_cosine_ndcg@10, before: 0.03563748625002177, after:0.74

In [32]:
with open(f'results/{ckpt_path.split("/")[-1]}.json', 'w') as f:
    f.write(json.dumps(scores_dict) + '\n' + args.to_json_string())

In [27]:
scores_dict

{'eq_to_category_test_cosine_accuracy@1_before': 44.44,
 'eq_to_category_test_cosine_accuracy@1_after': 77.78,
 'eq_to_category_test_cosine_accuracy@3_before': 88.89,
 'eq_to_category_test_cosine_accuracy@3_after': 100.0,
 'eq_to_category_test_cosine_accuracy@5_before': 100.0,
 'eq_to_category_test_cosine_accuracy@5_after': 100.0,
 'eq_to_category_test_cosine_accuracy@10_before': 100.0,
 'eq_to_category_test_cosine_accuracy@10_after': 100.0,
 'eq_to_category_test_cosine_precision@1_before': 44.44,
 'eq_to_category_test_cosine_precision@1_after': 77.78,
 'eq_to_category_test_cosine_precision@3_before': 62.96,
 'eq_to_category_test_cosine_precision@3_after': 85.19,
 'eq_to_category_test_cosine_precision@5_before': 55.56,
 'eq_to_category_test_cosine_precision@5_after': 75.56,
 'eq_to_category_test_cosine_precision@10_before': 43.33,
 'eq_to_category_test_cosine_precision@10_after': 60.0,
 'eq_to_category_test_cosine_recall@1_before': 2.96,
 'eq_to_category_test_cosine_recall@1_after': 12